Ohk, most of the approaches have problems with more dimensions.  

- Sparse dictionary: it ends up with overlaps. Im assuming there are certain patterns for components to come in. The sparse dictionary is trying to get a combinatorial of all patterns in a single atom, instead of having different atoms contain combinatorial of self contained values.
- Disjoint loss: its not working too.
  - too unstable to changes (hyperparams are very hard to crack)
  - "its not able to do patterns within". That is, if it gets a new pattern, it kind of goes wonk.
  - We might do something like: first train the disjoint loss encoder, then train a dictionary learning algo on the dimensions this one owns.
  - Does that make sense? This is one approach.
  - It will hopefully give us "top level patterns"
  - Problem is, if the whole thing is a top level pattern, in this case, it would keep decomposing until nothing is left
  - and lets not forget, it has trouble training too ;_;
  - In general, the problem is coming from the models not training at all i think. I had okayish runs in JAX for layers.2 actually, it had okay losses. lets see if any of them are alive.


- think on what stock approach dictionary learning can get for improvement
  - more data and more components -> so that we have strong sparsity -> only one person alive?

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path
import torch
from pt_to_api.benchmark.scalers import NormaliseStdScaler
import pt_to_api.benchmark.jax as JX
import pt_to_api.benchmark.torch as TX
from pathlib import Path
import torch
import json

import matplotlib.pyplot as plt
from pt_to_api.utils import show_single_channel_red_green_black as S
from pt_to_api.benchmark.anneal import CosineAnnealReconError

import matplotlib.pyplot as plt
import jax
import numpy as np
import jax.numpy as jnp
import gc
from joblib import Memory




In [ ]:
def get_weights_and_patches(layer_data_dir):
  weight = torch.load(layer_data_dir / "weight.pt", weights_only=False)
  patches = torch.load(layer_data_dir / "samples.pt", weights_only=False)
  return weight, patches

def save_normaliser_state(file_path, normaliser):
  with open(file_path, "w") as f:
    data = {
        "global_std_" :float(normaliser.global_std_),
    }
    json.dump(data, f)

def load_normaliser_state(file_path):
  with open(file_path, "r") as f:
    return json.load(f)


def get_loaded_normaliser(file_path):
  normaliser = NormaliseStdScaler()
  state = load_normaliser_state(file_path)
  normaliser.global_std_ = state["global_std_"]
  return normaliser


class RunManager:
  def __init__(self, dump_dir, n_components, seed_count):
    self.dump_dir = dump_dir
    self.n_components = n_components
    self.seed_count = seed_count

  def is_already_done(self):
    for p in self.get_save_paths():
      if not p.exists():
        return False
    return True

  @property
  def components_dump_dir(self):
    return self.dump_dir / f"{self.n_components}"

  def purge(self):
    files = list(self.components_dump_dir.glob("*.pt"))
    print(f"PURGE: deleting existing files for run {self.dump_dir=} {self.n_components=}")
    for f in files:
      f.unlink()

  def save_runs(self, runs):
    paths = self.get_save_paths()
    for i in range(len(runs)):
      paths[i].parent.mkdir(exist_ok=True, parents=True)
      torch.save(runs[i], paths[i])
    return paths

  def get_save_paths(self):
    return [self.components_dump_dir / f"seed_{seed}.pt" for seed in range(self.seed_count)]

In [ ]:
from pt_to_api.benchmark.init_strats import NoInitStrategy
from pt_to_api.benchmark.anneal import ConstantReconError
import pt_to_api.benchmark.torch as TX
def train_for_n_components(
    dump_dir,
    tensorboard_log_dir,
    scaled_pw,
    n_components,
    n_models,
    epochs,
    baseline_epochs,
    batch_size,
    force=False,
    train_kwargs=None,
):
    print(f"####################### {n_components =} ###########################")
    manager = RunManager(dump_dir, n_components, n_models)
    if force:
        manager.purge()
    if manager.is_already_done():
        print(f"SKIP: {n_components=} is already done for {n_models=}")
    if train_kwargs is None:
        train_kwargs = {}
    train_kwargs = {
        # "recon_err_schedule":CosineAnnealReconError(1000, min_factor=1, hold_frac=0),
        "init_strategy":NoInitStrategy(),
        "lr": 1e-2,
        **train_kwargs,
    }

    runs = JX.train(
        scaled_pw,
        n_components,
        n_models,
        epochs=epochs,
        baseline_epochs=baseline_epochs,
        batch_size=batch_size,
        tensorboard_log_dir=tensorboard_log_dir,
        # recon_err_schedule=CosineAnnealReconError(1000, min_factor=1, hold_frac=0),
        # init_strategy=NoInitStrategy(),
        **train_kwargs,
    )
    print(f"DONE: {n_components=}")
    return manager.save_runs(runs)
    


def main_training_cycle(
    data_dir,
    layer_name,
    channel,
    n_components_list,
    n_seeds,
    epochs=2000,
    baseline_epochs=1000,
    batch_size=512,
    force=False,
    train_kwargs=None,
    n_samples=1000,
):
    layer_data_dir = data_dir / layer_name / str(channel)
    weight, patches = get_weights_and_patches(layer_data_dir)
    
    scaler = NormaliseStdScaler().fit(patches)
    scaled_pw = scaler.transform(patches)

    # pw = patches * weight
    # scaler = NormaliseStdScaler().fit(pw)
    # scaled_pw = scaler.transform(pw)
    # scaled_pw = scaled_pw[:n_samples]
    # print("number of samples:", scaled_pw.shape[0])
    
    save_normaliser_state(layer_data_dir / "normaliser.json", scaler)
    dump_dir = layer_data_dir / "pos-only-runs"
    base_tensorboard_log_dir = layer_data_dir / "pos-only-tensorboard"

    run_paths = []
    for n_components in n_components_list:
        tensorboard_log_dir = base_tensorboard_log_dir / f"{n_components}"
        run_paths.append(train_for_n_components(
            dump_dir, tensorboard_log_dir, scaled_pw, n_components, n_seeds, epochs, baseline_epochs, batch_size, force, train_kwargs
        ))
    gc.collect()
    return run_paths

## Scratch

In [ ]:
# trainings not the best hmmmmmmmm.  
# i need to analyse the grads right now. 
# it is much easier in pytorch btw lol.
# but i would then need metrics work in pytorch also

In [ ]:
DATA_DIR = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist/collect-patches/data")

layer_data_dir = DATA_DIR / "layers.0" / str(0)
weight, patches = get_weights_and_patches(layer_data_dir)
pw = patches * weight
scaler = NormaliseStdScaler().fit(pw)
scaled_pw = scaler.transform(pw)
scaled_pw = scaled_pw[:1000]
print("number of samples:", scaled_pw.shape[0])



In [ ]:
from pt_to_api.benchmark.jax.init_models import parallel_init_models
from pt_to_api.benchmark.init_strats import NoInitStrategy
from pt_to_api.benchmark.hyperparams import get_scaled_hyperparamters
from pt_to_api.benchmark.jax import train_ as JXT

keys = jax.random.split(jax.random.PRNGKey(0), 5)

hp = get_scaled_hyperparamters(scaled_pw.std(), 9, 3, 1e-1)
models, optimizers = parallel_init_models(
    keys, 9, 3, hp, NoInitStrategy(), 1e-2
)


In [ ]:
# models.decoder.kernel.value != 0

In [ ]:
(models.decoder.kernel.value != 0).shape

In [ ]:
models.decoder.kernel.value.shape

In [ ]:
jnp.where(models.decoder.kernel.value != 0)

In [ ]:
models.decoder.kernel.value[models.decoder.kernel.value != 0]

In [ ]:
# from functools import partial
# get_uncond = partial(
#     JXT.ModelTrainStepUnconditionalParams,
#     sigma_eps=hp["sigma_eps"],
#     sigma_0=hp["sigma_0"],
#     sigma_s=hp["sigma_s"],
#     alpha=hp["alpha"],
# )


# uncond_params = get_uncond(
#     recon_err_multiplier=1, epoch_mod=1
# )

# x_jax = jnp.array(scaled_pw)

In [ ]:
# _, batch = next(JXT.get_batches(x_jax, 4, None))


In [ ]:
# grads = JXT.parallel_train_step(
#     models, optimizers, batch, uncond_params, False, "random"
# )

In [ ]:
# from flax.traverse_util import flatten_dict
# from flax import nnx

# flat_grads = flatten_dict(nnx.to_pure_dict(grads[0]), sep="/")
# for v in flat_grads["decoder/kernel"].mean(axis=1).mean(axis=1):
#     print(v)

In [ ]:
# from flax.traverse_util import flatten_dict
# from flax import nnx

# flat_grads = flatten_dict(nnx.to_pure_dict(grads), sep="/")

In [ ]:
# grads

# Dict learn


Let me also do dictionary learning then. We will save each dict in our data dir too.   
Its generally going to be transparent toi the autoencoder, we ll simply create a wrapper around the autoencoder maybe which works like sklearn? lets see.  i def need positive codes

In [ ]:
memory = Memory(location=".cache/learn_for_channel", verbose=1)

In [ ]:
from sklearn.decomposition import DictionaryLearning, MiniBatchDictionaryLearning
from typing import Literal

DICT_LEARN_CACHE = {}

def single_dict_learn(X, n_components, alg: Literal["minibatch", "basic"], seed=None, **kwargs):
    clz = DictionaryLearning if alg == "basic" else MiniBatchDictionaryLearning
    dl = clz(
        n_components=n_components, random_state=seed, **kwargs
    )
    dl.fit(X)
    codes = dl.transform(X)
    error = np.mean((X - codes @ dl.components_) ** 2)
    return dl, codes, error

@memory.cache
def do_dict_learn_across_seeds(data_fetcher, n_components, alg: Literal["minibatch", "basic"] = "basic", n_init=5, train_kwargs=None):
    X = data_fetcher.fetch_data()
    if train_kwargs is None:
        train_kwargs = {}
    best_dl, best_codes, best_error = None, None, np.inf
    all_dls = []
    for seed in range(n_init):
        dl, codes, error = single_dict_learn(X, n_components, alg, seed, **train_kwargs)
        all_dls.append(dl)
        if error < best_error:
            best_dl, best_codes, best_error = dl, codes, error
    return best_dl, best_codes, best_error, all_dls


In [ ]:
DATA_DIR = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist/collect-patches/data")

In [ ]:
import math
from tqdm import tqdm
from pt_to_api.utils import scatter_plot_1d
from pt_to_api.benchmark.utils import hungarian_match


from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ChannelDataFetcher:
    layer_data_dir: Path
    data_type: str
    n_samples: int

    def fetch_data(self):
        scaled_pw = _load_data_and_save_scaler(self.layer_data_dir, self.data_type)
        return scaled_pw[:self.n_samples]


def get_sparse_codes_ratio_with_more_than_1_active_node(sparse_codes, alpha_for_code_below=0.05):
    idxes = []
    for i in range(len(sparse_codes)):
        non_zero_codes = sparse_codes[i][sparse_codes[i] != 0]
        if len(non_zero_codes) == 0:
            continue
        non_zero_codes = np.abs(non_zero_codes)
        max_val_idx = np.argmax(non_zero_codes)
        thresh = non_zero_codes[max_val_idx] * alpha_for_code_below

        gt_thresh = []
        for j in range(len(non_zero_codes)):
            if j == max_val_idx:
                continue
            if non_zero_codes[j] > thresh:
                gt_thresh.append(non_zero_codes[j])
        if len(gt_thresh) > 0:
            idxes.append(i)

    return len(idxes) / sparse_codes.shape[0]

def _load_data_and_save_scaler(layer_data_dir, data_type="pw"):
    weight, patches = get_weights_and_patches(layer_data_dir)
    norm_state_path = "dict_learn_normaliser_for_patches.json" if data_type == "patches" else "dict_learn_normaliser.json"
    data = patches if data_type == "patches" else (patches*weight)
    scaler = NormaliseStdScaler().fit(data)
    save_normaliser_state(layer_data_dir / norm_state_path, scaler)
    scaled_pw = scaler.transform(data)
    
    return scaled_pw

def _calc_for_all(data_fetcher, comp_list, learn_kwargs=None):
    res = {}
    ns = []
    errors = []
    stab_scores = []
    for n in tqdm(comp_list):
        dl, codes, error, all_dls = do_dict_learn_across_seeds(data_fetcher, n, "minibatch", 5, learn_kwargs)
        _, stability_score, most_similar_idx, _ = hungarian_match([r.components_ for r in all_dls])
        stab_scores.append(stability_score)
        
        errors.append(error)
        ns.append(n)
        res[n] = {"dl": dl, "codes": codes, "error": error, "stability_score": stability_score}
        # res[n] = (dl, codes, error, stability_score)

    _, axs = plt.subplots(1, 2)
    axs[0].plot(ns, errors)
    axs[0].set_title("Error")
    axs[1].plot(ns, stab_scores)
    axs[1].set_title("stability score")
    plt.show()
    return res
    

def learn_for_channel(data_dir, layer_name, channel, comp_list, data_type="pw", n_samples=3000, learn_kwargs=None):
    layer_data_dir = DATA_DIR / layer_name / str(channel)
    data_fetcher = ChannelDataFetcher(layer_data_dir, data_type, n_samples)
    return data_fetcher.fetch_data(), _calc_for_all(data_fetcher, comp_list, learn_kwargs)


def show_dl(dl, codes, X, samples_to_show=10, image_shape=(3,3),row_sz=3, col_sz=3,show_scatter_plots=True):
    print("ratio of samples with >1 components active", get_sparse_codes_ratio_with_more_than_1_active_node(codes))
    components = dl if isinstance(dl, np.ndarray) else dl.components_
    if show_scatter_plots:
        print("##################### scatter plots for comps ##########################")
        for comp_idx in range(components.shape[0]):
            scatter_plot_1d(codes[:, comp_idx])
    print("##################### components ########################################")
    ncols = min(len(components), 8)
    nrows = math.ceil(len(components) / ncols)
    figsize = (ncols*col_sz, nrows*row_sz)
    S([c.reshape(image_shape) for c in components], figsize, ncols)
    plt.show()

    print("##################### x vs x_hat ########################################")
    recon = codes @ components
    n = min(samples_to_show, len(recon))
    for i in range(n):
        print(codes[i])
        S([X[i].reshape(image_shape), recon[i].reshape(image_shape)], 3, ax_titles=["original", "recon"])
        plt.show()
    

## layers.0

In [ ]:
COMP_LIST = [4,8,10,15,20,25,30]
DATA_DIR = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist/collect-patches/data")
layer_name = "layers.0"

### 0 ✅

We go with n-components=3. Easy

Ohk, so if i use PW mults, then the search space is smaller. its easier to fit also. for now though, im able to manage with normal patches too. quite interesting.  
mmmmmm, needs to thimks?   

it might make sense to go with patches though, it would find me interesting things (although, i still am very scared of finding 0s in some places).  


Ohk, i now have a feel of the scale of the problem lol :)     
The total number of patterns created by a kernel would be ridiculously high. this does make sense. but im at 50 for 9.   
And the curve is still not flattening.   

This seems true for both patches and pws. Now this would become crazy the moment we have more components 😶‍🌫️    
hmmmm hmmmmmmmmmmmmmm hmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmm.   



What would happen in my bigger cases? it would be brutal bro. My older intuition of having groups makes more sense now xD.  

In [ ]:
channel = 0
layer_data_dir = DATA_DIR / layer_name / str(channel)
data_type = "patches"
scaled_pw, res = learn_for_channel(DATA_DIR, "layers.0", channel, [5,10, 40,50], data_type=data_type)

In [ ]:
dl, codes = res[50]
show_dl(dl, codes, scaled_pw)

In [ ]:
# save the dl
dl, codes = res[40]
dl_save_path_key = "patches-dict-learning.pt" if data_type == "patches" else "dict-learning.pt"
print("saving to", dl_save_path_key)
torch.save({"dl": dl, "codes": codes}, layer_data_dir / dl_save_path_key)

In [ ]:
dl, codes = res[50]
show_dl(dl, codes, scaled_pw)

### 1 ✅

In [ ]:
channel = 1
layer_data_dir = DATA_DIR / layer_name / str(channel)
data_type = "patches"
scaled_pw, res = learn_for_channel(DATA_DIR, "layers.0", channel, [5,10,20,30,40,50], data_type=data_type)

In [ ]:
# save the dl
dl, codes = res[20]
dl_save_path_key = "patches-dict-learning.pt" if data_type == "patches" else "dict-learning.pt"
print("saving to", dl_save_path_key)
torch.save({"dl": dl, "codes": codes}, layer_data_dir / dl_save_path_key)

In [ ]:
dl, codes = res[4]
show_dl(dl, codes, scaled_pw)

### 2 ✅

In [ ]:
gc.collect()
channel = 2
layer_data_dir = DATA_DIR / layer_name / str(channel)
data_type = "patches"
scaled_pw, res = learn_for_channel(DATA_DIR, "layers.0", channel, [5,10,20,30,40,50], data_type=data_type)

In [ ]:
# save the dl
dl, codes = res[50]
dl_save_path_key = "patches-dict-learning.pt" if data_type == "patches" else "dict-learning.pt"
print("saving to", dl_save_path_key)
torch.save({"dl": dl, "codes": codes}, layer_data_dir / dl_save_path_key)

In [ ]:
dl, codes = res[5]
show_dl(dl, codes, scaled_pw)

### 3 ✅

In [ ]:
gc.collect()
channel = 3
layer_data_dir = DATA_DIR / layer_name / str(channel)
data_type = "patches"
scaled_pw, res = learn_for_channel(DATA_DIR, "layers.0", channel, [5,10,20,30,40,50,60,70], data_type=data_type)

In [ ]:
# save the dl
dl, codes = res[60]
dl_save_path_key = "patches-dict-learning.pt" if data_type == "patches" else "dict-learning.pt"
print("saving to", dl_save_path_key)
torch.save({"dl": dl, "codes": codes}, layer_data_dir / dl_save_path_key)

In [ ]:
dl, codes = res[6]
show_dl(dl, codes, scaled_pw)

### 4 ✅

In [ ]:
gc.collect()
channel = 4
layer_data_dir = DATA_DIR / layer_name / str(channel)
data_type = "patches"
scaled_pw, res = learn_for_channel(DATA_DIR, "layers.0", channel, [5,10,20,30,40,50], data_type=data_type)

In [ ]:
# save the dl
dl, codes = res[20]
torch.save({"dl": dl, "codes": codes}, layer_data_dir / "dict-learning.pt")

In [ ]:
# save the dl
dl, codes = res[20]
dl_save_path_key = "patches-dict-learning.pt" if data_type == "patches" else "dict-learning.pt"
print("saving to", dl_save_path_key)
torch.save({"dl": dl, "codes": codes}, layer_data_dir / dl_save_path_key)

### 5 ✅

In [ ]:
gc.collect()
channel = 5
layer_data_dir = DATA_DIR / layer_name / str(channel)
data_type = "patches"
scaled_pw, res = learn_for_channel(DATA_DIR, "layers.0", channel, [5,10,20,25,30,40,50], data_type=data_type)

In [ ]:
# save the dl
dl, codes = res[25]
dl_save_path_key = "patches-dict-learning.pt" if data_type == "patches" else "dict-learning.pt"
print("saving to", dl_save_path_key)
torch.save({"dl": dl, "codes": codes}, layer_data_dir / dl_save_path_key)

In [ ]:
dl, codes = res[21]
show_dl(dl, codes, scaled_pw)

### 6 ✅

In [ ]:
gc.collect()
channel = 6
layer_data_dir = DATA_DIR / layer_name / str(channel)
data_type = "patches"
scaled_pw, res = learn_for_channel(DATA_DIR, "layers.0", channel, [5,10,13,], data_type=data_type)

In [ ]:
# save the dl
dl, codes = res[13]
dl_save_path_key = "patches-dict-learning.pt" if data_type == "patches" else "dict-learning.pt"
print("saving to", dl_save_path_key)
torch.save({"dl": dl, "codes": codes}, layer_data_dir / dl_save_path_key)

In [ ]:
dl, codes = res[8]
show_dl(dl, codes, scaled_pw)

### 7 ✅

In [ ]:
gc.collect()
channel = 7
layer_data_dir = DATA_DIR / layer_name / str(channel)
data_type = "patches"
scaled_pw, res = learn_for_channel(DATA_DIR, "layers.0", channel, [5,10,40,50,60], data_type=data_type)

In [ ]:
# save the dl
dl, codes = res[40]
dl_save_path_key = "patches-dict-learning.pt" if data_type == "patches" else "dict-learning.pt"
print("saving to", dl_save_path_key)
torch.save({"dl": dl, "codes": codes}, layer_data_dir / dl_save_path_key)

In [ ]:
dl, codes = res[6]
show_dl(dl, codes, scaled_pw)

## layers.2

In [ ]:
COMP_LIST = [4,8,10,15,20,25,30]
DATA_DIR = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist/collect-patches/data")
layer_name = "layers.2"
n_samples = 10_000

### 0

In [ ]:
channel = 0
layer_data_dir = DATA_DIR / layer_name / str(channel)
data_type = "patches"
print("layer data dir", layer_data_dir)
scaled_pw, res = learn_for_channel(DATA_DIR, "layers.2", channel, [25,40,50,60,70,80,90,100,120,150,200,225,500], data_type=data_type, n_samples=n_samples)

In [ ]:
# save the dl
r = res[70]
dl_save_path_key = "patches-dict-learning.pt" if data_type == "patches" else "dict-learning.pt"
print("saving to", dl_save_path_key)
torch.save({"dl": r["dl"], "codes": r["codes"], "error": r["error"], "stability_score": r["stability_score"]}, layer_data_dir / dl_save_path_key)

In [ ]:
import torch
import torch.nn as nn

class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, l1_coeff=1e-3, corr_coeff=1e-3):
        super().__init__()
        self.encoder = nn.Linear(input_dim, hidden_dim, bias=True)
        self.decoder = nn.Linear(hidden_dim, input_dim, bias=True)
        self.l1_coeff = l1_coeff
        self.corr_coeff = corr_coeff

    def forward(self, x):
        acts = torch.relu(self.encoder(x))
        # recon = self.decoder(acts)
        latent = acts.unsqueeze(-1) * self.decoder.weight.T.unsqueeze(0)
        recon = latent.sum(dim=1)
        return recon, acts, latent

    def loss(self, x, recon, acts, latent):
        # recon_loss = (x - recon).pow(2).mean()
        recon_loss = (x - recon).pow(2).sum(dim=-1).mean()
        sparsity_loss = self.l1_coeff * acts.abs().sum(dim=-1).mean()
        corr_loss = self.corr_coeff * _corr_mat_loss(latent)
        # sparsity_loss = self.l1_coeff * acts.abs().mean()
        return recon_loss + sparsity_loss + corr_loss, recon_loss, sparsity_loss, corr_loss

def _corr_mat_loss(latent):
    l = latent**2
    corr_mat = torch.einsum("bid,bjd->bij", l, l)
    tot_loss = torch.triu(corr_mat, diagonal=1)
    return tot_loss.sum(dim=-1).sum(dim=-1).mean()
    

def train(model, data, epochs=100, lr=1e-3, batch_size=256):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    dataset = torch.utils.data.TensorDataset(data)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        for (batch,) in loader:
            recon, acts, latent = model(batch)
            loss, recon_loss, sparsity_loss, corr_loss = model.loss(batch, recon, acts, latent)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                model.decoder.weight.data = nn.functional.normalize(
                    model.decoder.weight.data, dim=0
                )

        if epoch % 10 == 0:
            with torch.no_grad():
                recon, acts, latent = model(data)
                loss, recon_loss, sparsity_loss, corr_loss = model.loss(data, recon, acts, latent)
                l0 = (acts > 0).float().sum(dim=-1).mean()
            print(f"epoch {epoch:4d} | loss {loss.item():.4f} | "
                  f"recon {recon_loss.item():.4f} | "
                  f"sparsity {sparsity_loss.item():.4f} | "
                  f"L0 {l0.item():.1f} | "
                  f"Corr {corr_loss.item():.1f}")

    return model

# # --- training loop ---
# def train(model, data, epochs=100, lr=1e-3):
#     optimizer = torch.optim.Adam(model.parameters(), lr=lr)

#     for epoch in range(epochs):
#         recon, acts, latent = model(data)
#         loss, recon_loss, sparsity_loss, corr_loss = model.loss(data, recon, acts, latent)
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         # optional: keep decoder columns unit norm (common in SAE literature)
#         with torch.no_grad():
#             model.decoder.weight.data = nn.functional.normalize(
#                 model.decoder.weight.data, dim=0
#             )

#         if epoch % 10 == 0:
#             l0 = (acts > 0).float().sum(dim=-1).mean()
#             print(f"epoch {epoch:4d} | loss {loss.item():.4f} | "
#                   f"recon {recon_loss.item():.4f} | "
#                   f"sparsity {sparsity_loss.item():.4f} | "
#                   f"L0 {l0.item():.1f} | "
#                   f"Corr {corr_loss.item():.1f}")

#     return model




In [ ]:
pca = SparsePCA(20)
pca.fit(scaled_pw[:2000])

In [ ]:
codes = pca.transform(scaled_pw[:2000])

In [ ]:
pca.components_.shape

In [ ]:
ds = codes[0][:, None] * pca.components_

In [ ]:
S([c.reshape(8,9) for c in ds], 10, 5)

In [ ]:
pca.explained_variance_ratio_

In [ ]:
import gc
gc.collect()

In [ ]:
device = "mps"
model = SparseAutoencoder(scaled_pw.shape[1], 50, l1_coeff=0.3, corr_coeff=1).to(device)
model = train(model, torch.tensor(scaled_pw).to(device), epochs=600, lr=1e-2)

In [ ]:
res[70]["dl"].components_.shape, res[70]["codes"].shape

In [ ]:
model.decoder.weight.shape

In [ ]:
x = torch.tensor(scaled_pw)
recon, acts, _ = model(x.to(device))
codes = acts.clone().detach().cpu().numpy()
components = model.decoder.weight.clone().detach().cpu().numpy().T
# print((x - recon).pow(2).mean())

In [ ]:
show_dl(components, codes, scaled_pw, image_shape=(8,9), show_scatter_plots=False)

In [ ]:
r0 = res[70]
show_dl(r0["dl"], r0["codes"], scaled_pw, image_shape=(8,9), show_scatter_plots=False)

In [ ]:
for i in [70,80,90,100]:
    print(i, res[i]["error"])

In [ ]:
# lets do 70 only then, easy


# Start

In [ ]:
DEFAULT_N_COMP_LIST = [2,3,4,5,6,7]

In [ ]:
from pt_to_api.benchmark.utils import support_overlap_matrix_batched

def get_support_overlap_of_run(run):
    # run.codes: [samples, n-comps]
    # components: [n-comps, dims]
    # need: [s,c,d]
    latents = np.einsum("sc,cd->scd", run.codes, run.components)
    overlap_matrix = support_overlap_matrix_batched(latents, threshold=0.1)
    overlap_matrix = overlap_matrix.fill_diagonal_(0)

    # this is the mean support overlap

    return overlap_matrix

In [ ]:
# well this sucks. Lets init using dict learned components?
# hmmm, this definitely is not working, dictionary learning is not giving good results cuz of the problem i had noticed 
# (it needs to essentially do subspace clustering of sorts)
# how am i trying to do it? I say, we keep disjoint support, and if recons works, then stuff works

In [ ]:
from pathlib import Path
from pt_to_api.benchmark.init_strats import StandardInitStrategy
from pt_to_api.benchmark.anneal import CosineIncreaseReconError
import torch
# lets only do for one seed for now
layer_name, channel = "layers.0", 0
DATA_DIR = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist/collect-patches/data")

# hack,  1/5 sqrt (mean ish)
sigma_0 = 1
max_alpha =  5000
run_paths = main_training_cycle(
  DATA_DIR, layer_name, channel, [2], 1, batch_size=32, baseline_epochs=500, epochs=500, force=True, train_kwargs={
      "lr": 1e-3, 
      "recon_err_schedule": ConstantReconError(),
      # "weights_coeff_range":(0,1),
      # "recon_coeff_range": (0.9,1),
      # "codes_coeff_range": (0.9,1),

      "alpha_schedule": CosineIncreaseReconError(1, 1/max_alpha, 0),
      "init_strategy": StandardInitStrategy(),
      # "weights_algo": "corr",
      # "sigma_eps_override": 0.01,
      # "initialised_model": model,
      # "codes_loss_coeff": 10,
  }
)
print("run paths", run_paths)

In [ ]:
# theres not much difference nice
run = torch.load(run_paths[0][0], weights_only=False)
S([c.reshape(3,3) for c in run.components], 4)
plt.show()

In [ ]:
fetcher = ReducedDimChannelDataFetcher(DATA_DIR / "layers.0" / "0", "patches", 1000, (4,5,6,7,8))
d = fetcher.fetch_data()

In [ ]:
f2 = ChannelDataFetcher(DATA_DIR / "layers.0" / "0", "patches", 1000)
d2 = f2.fetch_data()

In [ ]:
# learn_for_channel(DATA_DIR, "layers.0", "0", [5,10,20,30], "patches")

layer_data_dir = DATA_DIR / layer_name / str(channel)
data_fetcher_last_dims = ReducedDimChannelDataFetcher(DATA_DIR / "layers.0" / "0", "patches", 1000, (4,5,6,7,8))
res_last_dims = _calc_for_all(data_fetcher_last_dims, [5,10,15,20,25,30])

In [ ]:
# learn_for_channel(DATA_DIR, "layers.0", "0", [5,10,20,30], "patches")

layer_data_dir = DATA_DIR / layer_name / str(channel)
data_fetcher_first_dims = ReducedDimChannelDataFetcher(DATA_DIR / "layers.0" / "0", "patches", 1000, (0,1,2,3))
res_first_dims = _calc_for_all(data_fetcher_first_dims, [5,10,15,20,25,30])

In [ ]:
comps1.shape, comps2.shape

In [ ]:
def _with_original_dims(components, total_dims, dims_in_this_comp):
    n_comps, our_dim_length = components.shape
    res = np.zeros((n_comps, total_dims), dtype=np.float32)
    for d in range(our_dim_length):
        target_dim = dims_in_this_comp[d]
        res[:, target_dim] = components[:, d]
    return res    

In [ ]:
comps1 = _with_original_dims(res_last_dims[20]["dl"].components_, 9, (4,5,6,7,8))
comps2 = _with_original_dims(res_first_dims[15]["dl"].components_, 9, (0,1,2,3))
comps = np.concat([comps1, comps2])

In [ ]:
from sklearn.decomposition import SparseCoder
fetcher = ChannelDataFetcher(DATA_DIR / "layers.0" / "0", "patches", 1000)

data = fetcher.fetch_data()
coder = SparseCoder(
    dictionary=comps,           # your fixed dictionary (n_components × n_features)
    transform_algorithm='lasso_lars',  # or omp, lasso_cd, threshold
    transform_alpha=0.5
)
codes = coder.transform(data)

In [ ]:
data.shape

In [ ]:
recon = codes @ comps

In [ ]:
((recon - data)**2).mean()

In [ ]:
codes[0]

In [ ]:
i = 8
print(codes[i])
S([recon[i].reshape(3,3),data[i].reshape(3,3)])

In [ ]:
recon[0], data[0]

In [ ]:
for i in range(10):
    S([comps2[i].reshape(1,-1), res_first_dims[15]["dl"].components_[i].reshape(1,-1)], (10,1))
    plt.show()

In [ ]:
# think this more

# i run DJL on the components. this is supposed to give me dimensions which are freely moving.
# when does that happen? when we minimise loss as much as we can (towards the baseline)
# it wont every be as good as baseline, we have dictionary learning for that
# it would start becoming degenerate after you increase n-components to an extent
# when i increase n-components, it would start making single cell weights. 
# when do i know that its a simple break vs single cell weight?
# too many questions here.  


In [ ]:
mdl = MiniBatchDictionaryLearning(n_components=comps.shape[0], dict_init=comps, fit_algorithm="cd")

In [ ]:
fetcher = ChannelDataFetcher(DATA_DIR / "layers.0" / "0", "patches", 1000)

data = fetcher.fetch_data()

codes = mdl.fit_transform(data)

In [ ]:
recon = codes @ mdl.components_
((data - recon)**2).mean()

In [ ]:
# is it nto able to fully complete this?
# now there are many things which are hard to guess at this point
# when i do a descent on DJL, then it gives me a decomposition for n-components
# well, im feeling despair again :)
# this kinda suckssssss
comps[0]

In [ ]:
for i in range(10):
    S([comps[i].reshape(1,-1), res_last_dims[20]["dl"].components_[i].reshape(1,-1)], (10,1))
    plt.show()

In [ ]:
for i in range(10):
    S([comps1[i].reshape(1,-1), res_last_dims[20]["dl"].components_[i].reshape(1,-1)], (10,1))
    plt.show()

In [ ]:
# 15 is good.

res[15]["dl"]

In [ ]:
import math
from tqdm import tqdm
from pt_to_api.utils import scatter_plot_1d
from pt_to_api.benchmark.utils import hungarian_match


from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ReducedDimChannelDataFetcher:
    layer_data_dir: Path
    data_type: str
    n_samples: int
    dims_list: tuple[int,...]

    def fetch_data(self):
        scaled_pw = _load_data_and_save_scaler(self.layer_data_dir, self.data_type)
        dims_list = np.array(list(self.dims_list))
        return scaled_pw[:self.n_samples, dims_list]


# def get_sparse_codes_ratio_with_more_than_1_active_node(sparse_codes, alpha_for_code_below=0.05):
#     idxes = []
#     for i in range(len(sparse_codes)):
#         non_zero_codes = sparse_codes[i][sparse_codes[i] != 0]
#         if len(non_zero_codes) == 0:
#             continue
#         non_zero_codes = np.abs(non_zero_codes)
#         max_val_idx = np.argmax(non_zero_codes)
#         thresh = non_zero_codes[max_val_idx] * alpha_for_code_below

#         gt_thresh = []
#         for j in range(len(non_zero_codes)):
#             if j == max_val_idx:
#                 continue
#             if non_zero_codes[j] > thresh:
#                 gt_thresh.append(non_zero_codes[j])
#         if len(gt_thresh) > 0:
#             idxes.append(i)

#     return len(idxes) / sparse_codes.shape[0]

# def _load_data_and_save_scaler(layer_data_dir, data_type="pw"):
#     weight, patches = get_weights_and_patches(layer_data_dir)
#     norm_state_path = "dict_learn_normaliser_for_patches.json" if data_type == "patches" else "dict_learn_normaliser.json"
#     data = patches if data_type == "patches" else (patches*weight)
#     scaler = NormaliseStdScaler().fit(data)
#     save_normaliser_state(layer_data_dir / norm_state_path, scaler)
#     scaled_pw = scaler.transform(data)
    
#     return scaled_pw

# def _calc_for_all(data_fetcher, comp_list, learn_kwargs=None):
#     res = {}
#     ns = []
#     errors = []
#     stab_scores = []
#     for n in tqdm(comp_list):
#         dl, codes, error, all_dls = do_dict_learn_across_seeds(data_fetcher, n, "minibatch", 5, learn_kwargs)
#         _, stability_score, most_similar_idx, _ = hungarian_match([r.components_ for r in all_dls])
#         stab_scores.append(stability_score)
        
#         errors.append(error)
#         ns.append(n)
#         res[n] = {"dl": dl, "codes": codes, "error": error, "stability_score": stability_score}
#         # res[n] = (dl, codes, error, stability_score)

#     _, axs = plt.subplots(1, 2)
#     axs[0].plot(ns, errors)
#     axs[0].set_title("Error")
#     axs[1].plot(ns, stab_scores)
#     axs[1].set_title("stability score")
#     plt.show()
#     return res
    

# def learn_for_channel(data_dir, layer_name, channel, comp_list, data_type="pw", n_samples=3000, learn_kwargs=None):
#     layer_data_dir = DATA_DIR / layer_name / str(channel)
#     data_fetcher = ChannelDataFetcher(layer_data_dir, data_type, n_samples)
#     return data_fetcher.fetch_data(), _calc_for_all(data_fetcher, comp_list, learn_kwargs)


# def show_dl(dl, codes, X, samples_to_show=10, image_shape=(3,3),row_sz=3, col_sz=3,show_scatter_plots=True):
#     print("ratio of samples with >1 components active", get_sparse_codes_ratio_with_more_than_1_active_node(codes))
#     components = dl if isinstance(dl, np.ndarray) else dl.components_
#     if show_scatter_plots:
#         print("##################### scatter plots for comps ##########################")
#         for comp_idx in range(components.shape[0]):
#             scatter_plot_1d(codes[:, comp_idx])
#     print("##################### components ########################################")
#     ncols = min(len(components), 8)
#     nrows = math.ceil(len(components) / ncols)
#     figsize = (ncols*col_sz, nrows*row_sz)
#     S([c.reshape(image_shape) for c in components], figsize, ncols)
#     plt.show()

#     print("##################### x vs x_hat ########################################")
#     recon = codes @ components
#     n = min(samples_to_show, len(recon))
#     for i in range(n):
#         print(codes[i])
#         S([X[i].reshape(image_shape), recon[i].reshape(image_shape)], 3, ax_titles=["original", "recon"])
#         plt.show()
    

In [ ]:
from pathlib import Path
from pt_to_api.benchmark.init_strats import StandardInitStrategy
from pt_to_api.benchmark.anneal import CosineIncreaseReconError
import torch
# lets only do for one seed for now
layer_name, channel = "layers.0", 0
DATA_DIR = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist/collect-patches/data")

# hack,  1/5 sqrt (mean ish)
sigma_0 = 1
max_alpha =  5000
run_paths = main_training_cycle(
  DATA_DIR, layer_name, channel, [1], 1, batch_size=32, baseline_epochs=500, epochs=500, force=True, train_kwargs={
      "lr": 1e-3, 
      "recon_err_schedule": ConstantReconError(),
      # "weights_coeff_range":(0,1),
      # "recon_coeff_range": (0.9,1),
      # "codes_coeff_range": (0.9,1),

      "alpha_schedule": CosineIncreaseReconError(1, 1/max_alpha, 0),
      "init_strategy": StandardInitStrategy(),
      # "weights_algo": "corr",
      # "sigma_eps_override": 0.01,
      # "initialised_model": model,
      # "codes_loss_coeff": 10,
  }
)

In [ ]:
run = torch.load(run_paths[0][0], weights_only=False)
S([c.reshape(3,3) for c in run.components], 2)
plt.show()

In [ ]:
# lets only do for one seed for now
layer_name, channel = "layers.0", 5
DATA_DIR = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist/collect-patches/data")

main_training_cycle(
  DATA_DIR, layer_name, channel, DEFAULT_N_COMP_LIST, 5, batch_size=32, baseline_epochs=500, epochs=1000
)

In [ ]:


# lets only do for one seed for now
layer_name, channel = "layers.0", 2
DATA_DIR = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist/collect-patches/data")

main_training_cycle(
  DATA_DIR, layer_name, channel, DEFAULT_N_COMP_LIST, 5, batch_size=32, baseline_epochs=500, epochs=1000
)

In [ ]:
# lets only do for one seed for now
layer_name, channel = "layers.0", 3
DATA_DIR = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist/collect-patches/data")

main_training_cycle(
  DATA_DIR, layer_name, channel, DEFAULT_N_COMP_LIST, 5, batch_size=32, baseline_epochs=500, epochs=1000
)

In [ ]:
layer_name, channel = "layers.0", 4
DATA_DIR = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist/collect-patches/data")

main_training_cycle(
  DATA_DIR, layer_name, channel, DEFAULT_N_COMP_LIST, 5, batch_size=32, baseline_epochs=500, epochs=1000
)

In [ ]:
layer_name, channel = "layers.0", 5
DATA_DIR = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist/collect-patches/data")

main_training_cycle(
  DATA_DIR, layer_name, channel, DEFAULT_N_COMP_LIST, 5, batch_size=32, baseline_epochs=500, epochs=1000
)

In [ ]:
layer_name, channel = "layers.0", 6
DATA_DIR = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist/collect-patches/data")

main_training_cycle(
  DATA_DIR, layer_name, channel, DEFAULT_N_COMP_LIST, 5, batch_size=32, baseline_epochs=500, epochs=1000
)

In [ ]:
layer_name, channel = "layers.0", 7
DATA_DIR = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist/collect-patches/data")

main_training_cycle(
  DATA_DIR, layer_name, channel, DEFAULT_N_COMP_LIST, 5, batch_size=32, baseline_epochs=500, epochs=1000
)